# ⛏️ SAT Token GPU Miner (BSC)
Mining SAT token via keccak-256 PoW on Google Colab T4 GPU

**Contract:** `0x14Dc4b4929c664534f1d4D64107d8F36CbF906a0`
**Wallet:** `0x5a5fA9e1e108a2F5b047AdaeB94f28fD0708495d`
**Reward:** 50 SAT/block

In [ ]:
#@title ⛏️ GPU Miner {display-mode: "form"}
import torch
import time, os
from web3 import Web3
from web3.middleware import ExtraDataToPOAMiddleware as POA
from eth_account import Account
from Crypto.Hash import keccak as kc

# Generate NEW wallet at runtime (never touches GitHub)
acct = Account.create()
print(f"🔑 NEW WALLET (generated fresh)")
print(f"Address: {acct.address}")
print(f"Private Key: {acct.key.hex()}")
print(f"⚠️ SAVE THE KEY ABOVE! Send BNB to the address.")
print()

# Config
RPC = 'https://bsc-dataseed.bnbchain.org'
CA = '0x14Dc4b4929c664534f1d4D64107d8F36CbF906a0'
ABI = [
    {"inputs": [], "name": "challengeNumber", "outputs": [{"type": "bytes32"}], "stateMutability": "view", "type": "function"},
    {"inputs": [], "name": "miningTarget", "outputs": [{"type": "uint256"}], "stateMutability": "view", "type": "function"},
    {"inputs": [], "name": "getMiningReward", "outputs": [{"type": "uint256"}], "stateMutability": "view", "type": "function"},
    {"inputs": [{"name": "nonce", "type": "uint256"}, {"name": "challengeDigest", "type": "bytes32"}], "name": "mint", "outputs": [{"type": "bool"}], "stateMutability": "nonpayable", "type": "function"},
]

# Connect
w3 = Web3(Web3.HTTPProvider(RPC))
w3.middleware_onion.inject(POA, layer=0)
contract = w3.eth.contract(address=Web3.to_checksum_address(CA), abi=ABI)

challenge = contract.functions.challengeNumber().call()
target = contract.functions.miningTarget().call()
reward = contract.functions.getMiningReward().call()

print(f"Challenge: {challenge.hex()}")
print(f"Target: {target}")
print(f"Reward: {reward/10**8:.2f} SAT")

# Mining loop
print("\n⛏️ Starting mining...")
total_hashes = solutions = 0
t0 = time.time()
nonce = int.from_bytes(os.urandom(32), 'big')
miner_b = bytes.fromhex(acct.address[2:].lower())
last_ch = challenge; last_stats = t0
BATCH = 100000

try:
    while True:
        now = time.time()
        if now - last_stats > 10:
            try:
                nc = contract.functions.challengeNumber().call()
                if nc != last_ch:
                    print(f"\n🔄 New challenge: {nc.hex()}")
                    last_ch = nc; challenge = nc; nonce = int.from_bytes(os.urandom(32), 'big')
            except: pass
            if now - last_stats > 30:
                hr = total_hashes / max(1, now - t0)
                elapsed = (now - t0) / 60
                print(f"📊 {total_hashes:,} hashes | {hr/1e6:.2f} MH/s | {solutions} sols | {elapsed:.1f}min")
                last_stats = now
        for n in range(nonce, nonce + BATCH):
            nb = n.to_bytes(32, 'big')
            d = kc.new(digest_bits=256, data=challenge + miner_b + nb).digest()
            di = int.from_bytes(d, 'big')
            total_hashes += 1
            if di <= target:
                solutions += 1
                print(f"\n🎉 SOLUTION! nonce={n}")
                print(f"   Digest: {d.hex()}")
                try:
                    bnb = w3.eth.get_balance(acct.address)
                    if bnb > w3.to_wei(0.0001, 'ether'):
                        tx = contract.functions.mint(n, d).build_transaction({
                            'from': acct.address,
                            'nonce': w3.eth.get_transaction_count(acct.address),
                            'gas': 200000,
                            'gasPrice': w3.eth.gas_price,
                            'chainId': 56
                        })
                        signed = w3.eth.account.sign_transaction(tx, acct.key)
                        txh = w3.eth.send_raw_transaction(signed.raw_transaction)
                        print(f"   TX: {txh.hex()}")
                    else:
                        print(f"   ⚠️ No BNB! Send to {acct.address}")
                except Exception as e:
                    print(f"   Err: {e}")
        nonce += BATCH
except KeyboardInterrupt:
    elapsed = time.time() - t0
    print(f"\n=== Mining stopped | {total_hashes:,} hashes | {total_hashes/max(1,elapsed):.0f} H/s | {solutions} sols | {elapsed/60:.1f}min ===")
